# B 노트북 · 규칙 기반 RFM, K-means, 가설 H1·H2

**담당:** B (이름: 서유나) · **중간 점검:** 11/29 · **최종 산출:** 규칙 기반 RFM 세그먼트 섹션, H1·H2 검정 결과와 해석

**이 노트북에서 할 일**
1. R·F·M을 5분위 점수로 바꾸고 세그먼트를 정의한다 (아래 기본 구현 있음)
2. H1: 세그먼트별 재구매율이 다른가 (카이제곱, Cramér's V), F·M이 R 이상의 정보를 주는가
3. H2: 세그먼트별 라벨 구간 매출이 다른가 (Kruskal-Wallis → Dunn, Bonferroni)
4. 보조: K-means 군집과 규칙 세그먼트의 일치도 (ARI)
5. 세 기준일(S1·S2·S3) 모두에서 반복해 안정성을 본다
6. 규칙 기반 점수를 `ru.to_score_frame`으로 내서 A·C와 같은 표에 올린다

고객은 **거래처 고객**이라고 쓴다.

In [1]:
import sys; sys.path.insert(0, "..")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import rfm_utils as ru

clean, report = ru.clean_transactions(ru.load_raw())
inv = ru.invoice_table(clean)
datasets = {name: ru.make_dataset(inv, snap) for name, snap in ru.SNAPSHOTS.items()}
for name, snap in ru.SNAPSHOTS.items():
    ru.assert_no_leakage(inv, datasets[name], snap)   # 새 데이터셋을 만들 때마다 호출
{k: len(v) for k, v in datasets.items()}   # 기준일별 고객 수

{'S1': 4537, 'S2': 4933, 'S3': 5249}

## 1. 5분위 점수와 세그먼트
R은 작을수록 좋으므로 점수를 뒤집는다. 같은 값이 많아 `rank(method="first")`를 먼저 쓴다.

In [2]:
def quintile(s, ascending=True):
    r = s.rank(method="first", ascending=ascending)
    return pd.qcut(r, 5, labels=[1, 2, 3, 4, 5]).astype(int)

def add_rfm_scores(d):
    d = d.copy()
    d["R_s"] = quintile(d["R"], ascending=False)   # 최근일수록 5
    d["F_s"] = quintile(d["F"])
    d["M_s"] = quintile(d["M"])
    d["rfm_sum"] = d["R_s"] + d["F_s"] + d["M_s"]
    return d

def segment(row):
    # 단순 예시 규칙. 팀 합의 후 바꿔도 된다 (바꾸면 본문에 규칙표를 넣는다)
    if row.R_s >= 4 and row.F_s >= 4: return "Champions"
    if row.R_s >= 3 and row.F_s >= 3: return "Loyal"
    if row.R_s >= 4: return "Recent"
    if row.R_s <= 2 and row.F_s >= 3: return "At risk"
    return "Lost / Low"

seg = {}
for name, d in datasets.items():
    d = add_rfm_scores(d)
    d["segment"] = d.apply(segment, axis=1)
    seg[name] = d
seg["S2"].groupby("segment").agg(customers=("repurchase", "size"), repurchase_rate=("repurchase", "mean"),
                                  revenue_mean=("future_revenue", "mean")).round(3)

,customers,repurchase_rate,revenue_mean
segment,,,
At risk,726,0.182,96.127
Champions,1271,0.692,1021.248
Lost / Low,1572,0.095,41.739
Loyal,963,0.347,195.405
Recent,401,0.312,135.827


## 2. H1 · 세그먼트별 재구매율 (카이제곱, Cramér's V)

In [3]:
from scipy.stats import chi2_contingency

def h1_test(d):
    table = pd.crosstab(d["segment"], d["repurchase"])
    chi2, p, dof, _ = chi2_contingency(table)
    v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
    return {"chi2": chi2, "p": p, "dof": dof, "cramers_v": v}

pd.DataFrame({name: h1_test(d) for name, d in seg.items()}).T.round(4)

,chi2,p,dof,cramers_v
S1,918.4539,0.0,4.0,0.4499
S2,1222.6202,0.0,4.0,0.4978
S3,1179.2739,0.0,4.0,0.4740


**TODO (B):** F·M이 R 이상의 정보를 주는가 → R만 쓴 5그룹 vs RFM 세그먼트의 재구매율 분리도를 비교한다 (예: 로지스틱 회귀에서 R만 vs R+F+M의 우도비 검정, 또는 R 점수 고정 후 F·M 효과).

## 3. H2 · 세그먼트별 라벨 구간 매출 (Kruskal-Wallis)

In [4]:
from scipy.stats import kruskal

def h2_test(d):
    groups = [g["future_revenue"].values for _, g in d.groupby("segment")]
    h, p = kruskal(*groups)
    return {"H": h, "p": p}

pd.DataFrame({name: h2_test(d) for name, d in seg.items()}).T.round(4)

,H,p
S1,1048.3642,0.0
S2,1368.8063,0.0
S3,1404.6932,0.0


**TODO (B):** 사후검정 Dunn + Bonferroni (`pip install scikit-posthocs`, `requirements.txt`에 버전 추가). 매출은 치우친 분포이므로 박스플롯은 로그 축으로 그린다.

## 4. 보조: K-means vs 규칙 세그먼트 (ARI)

In [5]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

d = seg["S2"]
X = StandardScaler().fit_transform(np.log1p(d[["R", "F", "M"]]))
km = KMeans(n_clusters=5, n_init=10, random_state=ru.SEED).fit(X)
print("ARI(K-means, rule segments):", round(adjusted_rand_score(d["segment"], km.labels_), 3))

/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/private/tmp/claude-501/-Users-changmin-Desktop-AI---------final-project/30771d6c-1b9f-4e24-ab9f-31ed5e2310af/scratchpad/venv/lib/python3.9/site-packages/sklearn/cluster/_kmeans.py:237: RuntimeWarning: divide by zero encountered in matmul
  current_pot =

ARI(K-means, rule segments): 0.38


## 5. 규칙 기반 점수를 공통 형식으로 내기
세그먼트는 숫자 점수로 바꿔서 낸다. 여기서는 `rfm_sum`(3~15)을 쓴다. 동점이 많아 `lift_at_k`가 무작위로 동점을 깬다.

In [6]:
rows = []
for name, d in seg.items():
    sf = ru.to_score_frame(d.index, d["rfm_sum"], ru.SNAPSHOTS[name], "rule_rfm")
    ru.validate_score_frame(sf, d)
    rows.append({"snapshot": name, **ru.evaluate_scores(d["repurchase"], d["rfm_sum"])})
pd.DataFrame(rows).round(3)

,snapshot,auc,pr_auc,base_rate,capture@10,precision@10,lift@10
0,S1,0.783,0.667,0.350,0.241,0.844,2.412
1,S2,0.805,0.668,0.329,0.255,0.838,2.549
2,S3,0.784,0.724,0.427,0.211,0.899,2.106


## 다음 단계 (B)
- [ ] 세그먼트 규칙 확정, 본문에 규칙표와 이유
- [ ] H1에서 F·M 추가 정보 검정
- [ ] H2 Dunn 사후검정과 효과 크기
- [ ] 세 기준일에서 세그먼트 구성·재구매율이 안정적인지 표로
- [ ] K-means k 선택 근거 (엘보·실루엣)
- [ ] 결과가 "기대와 달라도" 그대로 쓴다 (기획안 6장)

## 6. 결과 저장 (A에게 넘기기)
세 기준일의 규칙 기반 점수를 `outputs/scores_rule_rfm.csv`로 저장하고 커밋한다. 규칙을 바꿀 때마다 다시 실행해 덮어쓴다.

In [7]:
frames = [ru.to_score_frame(d.index, d["rfm_sum"], ru.SNAPSHOTS[name], "rule_rfm") for name, d in seg.items()]
print(ru.save_score_frame(pd.concat(frames, ignore_index=True), "rule_rfm"))

/Users/changmin/Desktop/AI데이터분석및실습/final project/outputs/scores_rule_rfm.csv
